In [0]:
import yaml
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import Task, NotebookTask, TaskDependency, Source, QueueSettings

# ── Load job configuration from YAML ────────────────────────────────────────
yaml_path = "/Workspace/Repos/jaysonheyyou@gmail.com/DATABRICKS-END-TO-END/yml/automation_job"
with open(yaml_path) as f:
    config = yaml.safe_load(f)

job_config = config["resources"]["jobs"]["automation_sales_etl_job"]

# ── Initialize Databricks client ───────────────────────────────────────────
w = WorkspaceClient()

# ── Build task list with dependencies ──────────────────────────────────────
tasks = []
for task_spec in job_config["tasks"]:
    # Build dependencies if they exist
    depends_on = None
    if "depends_on" in task_spec:
        depends_on = [
            TaskDependency(task_key=dep["task_key"]) 
            for dep in task_spec["depends_on"]
        ]
    
    # Create the task
    task = Task(
        task_key=task_spec["task_key"],
        notebook_task=NotebookTask(
            notebook_path=task_spec["notebook_task"]["notebook_path"],
            source=Source(task_spec["notebook_task"]["source"])
        ),
        depends_on=depends_on
    )
    tasks.append(task)

# ── Create the job ─────────────────────────────────────────────────────────
print(f"Creating job: {job_config['name']}...")

created_job = w.jobs.create(
    name=job_config["name"],
    tasks=tasks,
    queue=QueueSettings(enabled=job_config["queue"]["enabled"])
)

print(f"\n✓ Job created successfully!")
print(f"  Job ID: {created_job.job_id}")
print(f"  Job Name: {job_config['name']}")
print(f"  Tasks: {len(tasks)}")
print(f"\n  View job: https://dbc-21746fe0-c5fd.cloud.databricks.com/#job/{created_job.job_id}")